In [7]:


import os
import sys
import numpy as np
import pandas as pd
import optuna
import xgboost as xgb
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.linear_model import ElasticNetCV, RidgeCV
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
)
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, RobustScaler
import tensorflow as tf
from tensorflow import keras
from keras import layers



optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = 25  # Optuna trials per tunable model
RANDOM_STATE = 42

USE_IMPUTATION = True
WINSORIZE_QUANTILE = 0.99
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.80, 0.10, 0.10
USE_RANDOM_SPLIT = False
SPLIT_RANDOM_STATE = 42
SCALER_TYPE = "standard"

LAG_DAYS = [1, 3, 7]
ROLL_WINDOWS = [3, 7]
ADMISSION_LAG_7 = "admission_lag7"

TIMESTAMP_COL = "Timestamp"
DATE_COL = "Date"
TARGET_COL = "Number of Admissions"
WIND_DIR_COL = "wind_direction_10m_dominant (°)"

TIME_FEATURES = ["dow_sin", "dow_cos", "month_sin", "month_cos", "weekend"]


def get_raw_feature_columns(df):
    """Numeric air/weather columns only (exclude date, target, wind direction angle, and outcome-like columns)."""
    exclude = {DATE_COL, TARGET_COL, WIND_DIR_COL}
    outcome_like = ["brought_dead", "admission"]
    return [
        c
        for c in df.columns
        if c not in exclude
        and pd.api.types.is_numeric_dtype(df[c])
        and not any(kw in c.lower() for kw in outcome_like)
    ]


def get_feature_columns(df):
    """Model input: time features + _lag1/_lag3/_lag7, _roll3/_roll7, and admission_lag7 (no same-day raw)."""
    time_cols = [c for c in TIME_FEATURES if c in df.columns]
    lag_roll = [
        c
        for c in df.columns
        if any(c.endswith(f"_lag{k}") for k in LAG_DAYS)
        or any(c.endswith(f"_roll{k}") for k in ROLL_WINDOWS)
    ]
    admission_cols = [ADMISSION_LAG_7] if ADMISSION_LAG_7 in df.columns else []
    return time_cols + lag_roll + admission_cols


def load_data():
    """
    Load merged feature dataset (all_features_data_changed_2.csv),
    drop specified columns, and construct a Date column.
    """
    path = "/Users/suhaniagarwal/Downloads/all_features_data_changed_2.csv"
    if not os.path.exists(path):
        script_dir = os.path.dirname(os.path.abspath(__file__))
        path = os.path.join(script_dir, "all_features_data_changed_2.csv")

    df = pd.read_csv(path)

    if TIMESTAMP_COL in df.columns:
        df[TIMESTAMP_COL] = pd.to_datetime(df[TIMESTAMP_COL], errors="coerce")
        df[DATE_COL] = df[TIMESTAMP_COL].dt.normalize()
    elif DATE_COL in df.columns:
        df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

    drop_exact = [
        "wind_speed_10m_max (km/h)",
        "temperature_2m_min (°C)",
        "wind_gusts_10m_max (km/h)",
        "NO2 (µg/m³)",
        "Benzene (µg/m³)",
        "PM10 (µg/m³)",
        "NO (µg/m³)",
        "relative_humidity_2m_min (%)",
        "Toluene (µg/m³)",
        "Ozone (µg/m³)",
    ]
    df = df.drop(columns=[c for c in drop_exact if c in df.columns])

    nh3_cols = [c for c in df.columns if "NH3" in c]
    nox_cols = [c for c in df.columns if "NOx" in c]
    if nh3_cols or nox_cols:
        df = df.drop(columns=list(set(nh3_cols + nox_cols)))

    return df


def preprocess_data(df, impute_features=False):
    df = df.sort_values(DATE_COL).copy()
    df = df.dropna(subset=[TARGET_COL])

    raw_cols = get_raw_feature_columns(df)

    if impute_features:
        for col in raw_cols:
            if col in df.columns and df[col].isna().any():
                df[col] = df[col].ffill().bfill()

    df = df.dropna(subset=raw_cols)

    q = df[TARGET_COL].quantile(WINSORIZE_QUANTILE)
    df.loc[df[TARGET_COL] > q, TARGET_COL] = q

    return df.reset_index(drop=True)


def feature_engineering(df):
    """Add time features, wind direction encoding, 1/3/7-day lags and 3/7-day rolling means for air/weather, plus admission_lag7."""
    df = df.sort_values(DATE_COL).reset_index(drop=True)

    dt = pd.to_datetime(df[DATE_COL])
    df["dow_sin"] = np.sin(2 * np.pi * dt.dt.dayofweek / 7)
    df["dow_cos"] = np.cos(2 * np.pi * dt.dt.dayofweek / 7)
    df["month_sin"] = np.sin(2 * np.pi * dt.dt.month / 12)
    df["month_cos"] = np.cos(2 * np.pi * dt.dt.month / 12)
    df["weekend"] = (dt.dt.dayofweek >= 5).astype(np.float64)

    if WIND_DIR_COL in df.columns:
        df["wind_sin"] = np.sin(2 * np.pi * df[WIND_DIR_COL] / 360.0)
        df["wind_cos"] = np.cos(2 * np.pi * df[WIND_DIR_COL] / 360.0)

    raw_cols = get_raw_feature_columns(df)

    new_cols = {}
    lag_roll_cols = []
    for col in raw_cols:
        for k in LAG_DAYS:
            name = f"{col}_lag{k}"
            new_cols[name] = df[col].shift(k)
            lag_roll_cols.append(name)
        for w in ROLL_WINDOWS:
            name = f"{col}_roll{w}"
            new_cols[name] = df[col].rolling(w, min_periods=1).mean().shift(1)
            lag_roll_cols.append(name)

    new_cols[ADMISSION_LAG_7] = df[TARGET_COL].shift(7)
    lag_roll_cols.append(ADMISSION_LAG_7)

    df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)

    df = df.dropna(how="any", subset=lag_roll_cols)

    return df.reset_index(drop=True)


def _time_aware_split_indices(n, train_frac, val_frac, test_frac):
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)
    train_ix = np.arange(0, n_train)
    val_ix = np.arange(n_train, n_train + n_val)
    test_ix = np.arange(n_train + n_val, n)
    return train_ix, val_ix, test_ix


def _random_split_indices(n, train_frac, val_frac, test_frac, random_state=None):
    rng = np.random.default_rng(
        random_state if random_state is not None else SPLIT_RANDOM_STATE
    )
    perm = rng.permutation(n)
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)
    train_ix = perm[:n_train]
    val_ix = perm[n_train : n_train + n_val]
    test_ix = perm[n_train + n_val :]
    return train_ix, val_ix, test_ix


def training_data_split(
    df, train_frac=None, val_frac=None, test_frac=None, use_random_split=None
):
    train_frac = train_frac if train_frac is not None else TRAIN_FRAC
    val_frac = val_frac if val_frac is not None else VAL_FRAC
    test_frac = test_frac if test_frac is not None else TEST_FRAC
    use_random = use_random_split if use_random_split is not None else USE_RANDOM_SPLIT

    assert abs(train_frac + val_frac + test_frac - 1.0) < 1e-9

    feat_cols = [c for c in get_feature_columns(df) if c in df.columns]
    if not feat_cols:
        raise ValueError("No feature columns found in df.")

    X = df[feat_cols].astype(float)
    y = df[TARGET_COL]

    n = len(df)
    if use_random:
        train_ix, val_ix, test_ix = _random_split_indices(
            n, train_frac, val_frac, test_frac
        )
    else:
        train_ix, val_ix, test_ix = _time_aware_split_indices(
            n, train_frac, val_frac, test_frac
        )

    X_train = X.iloc[train_ix].copy()
    X_val = X.iloc[val_ix].copy()
    X_test = X.iloc[test_ix].copy()
    y_train = y.iloc[train_ix]
    y_val = y.iloc[val_ix]
    y_test = y.iloc[test_ix]

    scaler = RobustScaler() if SCALER_TYPE == "robust" else StandardScaler()
    X_train = pd.DataFrame(
        scaler.fit_transform(X_train), columns=feat_cols, index=X_train.index
    )
    X_val = pd.DataFrame(
        scaler.transform(X_val), columns=feat_cols, index=X_val.index
    )
    X_test = pd.DataFrame(
        scaler.transform(X_test), columns=feat_cols, index=X_test.index
    )

    return X_train, X_val, X_test, y_train, y_val, y_test, scaler


def evaluate_model(model, X, y):
    pred = model.predict(X)
    return {
        "mae": mean_absolute_error(y, pred),
        "rmse": np.sqrt(mean_squared_error(y, pred)),
        "r2": r2_score(y, pred),
    }


class XGBRegressorWrapper(xgb.XGBRegressor):
    """
    Thin wrapper around XGBRegressor to always work with NumPy arrays.

    This avoids pandas/xgboost dtype quirks (e.g. 'DataFrame' object has no attribute "dtype")
    by converting inputs to plain float32 arrays before fitting/predicting.
    """

    def fit(self, X, y, *args, **kwargs):
        X_arr = np.asarray(X, dtype=np.float32)
        y_arr = np.asarray(y, dtype=np.float32)
        return super().fit(X_arr, y_arr, *args, **kwargs)

    def predict(self, X, *args, **kwargs):
        X_arr = np.asarray(X, dtype=np.float32)
        return super().predict(X_arr, *args, **kwargs)


class _PredictorWrapper:
    def __init__(self, predict_fn):
        self.predict = predict_fn


def _build_keras_model(n_features, params, seed=42):
    keras.backend.clear_session()
    tf.random.set_seed(seed)
    model = keras.Sequential()
    model.add(keras.Input(shape=(n_features,)))
    reg = keras.regularizers.L2(params["l2"]) if params["l2"] > 0 else None
    for _ in range(params["n_layers"]):
        model.add(
            layers.Dense(
                params["units"],
                activation="relu",
                kernel_initializer="he_normal",
                kernel_regularizer=reg,
            )
        )
        model.add(layers.BatchNormalization())
        model.add(layers.Dropout(params["dropout"]))
    model.add(layers.Dense(1, kernel_regularizer=reg))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=params["lr"]),
        loss="mse",
        metrics=["mae"],
    )
    return model


def tune_and_fit_keras(X_train, y_train, X_val, y_val, X_test, y_test, n_trials=N_TRIALS):
    X_tr = np.asarray(X_train, dtype=np.float32)
    y_tr = np.asarray(y_train, dtype=np.float32).reshape(-1, 1)
    X_va = np.asarray(X_val, dtype=np.float32)
    y_va = np.asarray(y_val, dtype=np.float32).reshape(-1, 1)
    n_features = X_tr.shape[1]

    def objective(trial):
        params = {
            "units": trial.suggest_categorical("units", [16, 32, 64, 128]),
            "n_layers": trial.suggest_int("n_layers", 1, 3),
            "dropout": trial.suggest_float("dropout", 0.2, 0.5),
            "lr": trial.suggest_float("lr", 3e-4, 3e-3, log=True),
            "l2": trial.suggest_float("l2", 5e-5, 5e-3, log=True),
        }
        model = _build_keras_model(n_features, params, RANDOM_STATE)
        early = keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=15,
            restore_best_weights=True,
        )
        model.fit(
            X_tr,
            y_tr,
            validation_data=(X_va, y_va),
            epochs=250,
            batch_size=32,
            callbacks=[early],
            verbose=0,
        )
        pred = model.predict(X_va, verbose=0).ravel()
        return r2_score(y_val, pred)

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE, n_startup_trials=5),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = study.best_params
    best["units"] = int(best["units"])
    best["n_layers"] = int(best["n_layers"])
    final = _build_keras_model(n_features, best, RANDOM_STATE)
    early = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=20,
        restore_best_weights=True,
    )
    final.fit(
        X_tr,
        y_tr,
        validation_data=(X_va, y_va),
        epochs=250,
        batch_size=32,
        callbacks=[early],
        verbose=0,
    )
    wrap = _PredictorWrapper(
        lambda X: final.predict(np.asarray(X, dtype=np.float32), verbose=0).ravel()
    )
    return final, evaluate_model(wrap, X_val, y_val), evaluate_model(wrap, X_test, y_test)


def tune_and_fit_xgboost(X_train, y_train, X_val, y_val, X_test, y_test, n_trials=N_TRIALS):
    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 50, 300),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "learning_rate": trial.suggest_float(
                "learning_rate", 0.01, 0.2, log=True
            ),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float(
                "colsample_bytree", 0.6, 1.0
            ),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        }
        model = XGBRegressorWrapper(**params, random_state=RANDOM_STATE)
        model.fit(X_train, y_train)
        pred = model.predict(X_val)
        return r2_score(y_val, pred)

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE, n_startup_trials=5),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = study.best_params
    model = XGBRegressorWrapper(**best, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    return (
        model,
        evaluate_model(model, X_val, y_val),
        evaluate_model(model, X_test, y_test),
    )


def tune_and_fit_rf(X_train, y_train, X_val, y_val, X_test, y_test, n_trials=N_TRIALS):
    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 50, 200),
            "max_depth": trial.suggest_int("max_depth", 5, 20),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 2, 20),
        }
        model = RandomForestRegressor(
            **params, random_state=RANDOM_STATE, n_jobs=-1
        )
        model.fit(X_train, y_train)
        return r2_score(y_val, model.predict(X_val))

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE, n_startup_trials=5),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = study.best_params
    model = RandomForestRegressor(
        **best, random_state=RANDOM_STATE, n_jobs=-1
    )
    model.fit(X_train, y_train)
    return (
        model,
        evaluate_model(model, X_val, y_val),
        evaluate_model(model, X_test, y_test),
    )


def tune_and_fit_gbm(X_train, y_train, X_val, y_val, X_test, y_test, n_trials=N_TRIALS):
    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 50, 300),
            "max_depth": trial.suggest_int("max_depth", 2, 8),
            "learning_rate": trial.suggest_float(
                "learning_rate", 0.01, 0.2, log=True
            ),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 30),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        }
        model = GradientBoostingRegressor(
            **params, random_state=RANDOM_STATE
        )
        model.fit(X_train, y_train)
        return r2_score(y_val, model.predict(X_val))

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE, n_startup_trials=5),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = study.best_params
    model = GradientBoostingRegressor(
        **best, random_state=RANDOM_STATE
    )
    model.fit(X_train, y_train)
    return (
        model,
        evaluate_model(model, X_val, y_val),
        evaluate_model(model, X_test, y_test),
    )


def tune_and_fit_histgb(
    X_train, y_train, X_val, y_val, X_test, y_test, n_trials=N_TRIALS
):
    def objective(trial):
        params = {
            "max_iter": trial.suggest_int("max_iter", 50, 250),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "learning_rate": trial.suggest_float(
                "learning_rate", 0.01, 0.2, log=True
            ),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 30),
        }
        model = HistGradientBoostingRegressor(
            **params, random_state=RANDOM_STATE
        )
        model.fit(X_train, y_train)
        return r2_score(y_val, model.predict(X_val))

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE, n_startup_trials=5),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = study.best_params
    model = HistGradientBoostingRegressor(
        **best, random_state=RANDOM_STATE
    )
    model.fit(X_train, y_train)
    return (
        model,
        evaluate_model(model, X_val, y_val),
        evaluate_model(model, X_test, y_test),
    )


def tune_and_fit_extra_trees(
    X_train, y_train, X_val, y_val, X_test, y_test, n_trials=N_TRIALS
):
    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 50, 250),
            "max_depth": trial.suggest_int("max_depth", 5, 20),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 2, 15),
        }
        model = ExtraTreesRegressor(
            **params, random_state=RANDOM_STATE, n_jobs=-1
        )
        model.fit(X_train, y_train)
        return r2_score(y_val, model.predict(X_val))

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE, n_startup_trials=5),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = study.best_params
    model = ExtraTreesRegressor(
        **best, random_state=RANDOM_STATE, n_jobs=-1
    )
    model.fit(X_train, y_train)
    return (
        model,
        evaluate_model(model, X_val, y_val),
        evaluate_model(model, X_test, y_test),
    )


def tune_and_fit_svr(X_train, y_train, X_val, y_val, X_test, y_test, n_trials=N_TRIALS):
    def objective(trial):
        C = trial.suggest_float("C", 0.1, 100.0, log=True)
        epsilon = trial.suggest_float("epsilon", 0.01, 1.0)
        kernel = trial.suggest_categorical("kernel", ["rbf", "poly"])
        model = SVR(C=C, epsilon=epsilon, kernel=kernel)
        model.fit(X_train, y_train)
        return r2_score(y_val, model.predict(X_val))

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE, n_startup_trials=5),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = study.best_params
    model = SVR(**best)
    model.fit(X_train, y_train)
    return (
        model,
        evaluate_model(model, X_val, y_val),
        evaluate_model(model, X_test, y_test),
    )


def tune_and_fit_mlp(X_train, y_train, X_val, y_val, X_test, y_test, n_trials=N_TRIALS):
    def objective(trial):
        n_layers = trial.suggest_int("n_layers", 1, 3)
        layers_units = []
        for i in range(n_layers):
            layers_units.append(trial.suggest_int(f"units_{i}", 16, 128))
        alpha = trial.suggest_float("alpha", 1e-5, 1e-2, log=True)
        lr = trial.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True)
        model = MLPRegressor(
            hidden_layer_sizes=tuple(layers_units),
            activation="relu",
            solver="adam",
            alpha=alpha,
            learning_rate_init=lr,
            max_iter=1000,
            random_state=RANDOM_STATE,
        )
        model.fit(X_train, y_train)
        return r2_score(y_val, model.predict(X_val))

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE, n_startup_trials=5),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = study.best_params
    n_layers = best.pop("n_layers")
    units = [best.pop(f"units_{i}") for i in range(n_layers)]
    best["hidden_layer_sizes"] = tuple(units)
    best["activation"] = "relu"
    best["solver"] = "adam"
    best["max_iter"] = 1000
    best["random_state"] = RANDOM_STATE
    model = MLPRegressor(**best)
    model.fit(X_train, y_train)
    return (
        model,
        evaluate_model(model, X_val, y_val),
        evaluate_model(model, X_test, y_test),
    )


def main():
    df = load_data()
    n_load = len(df)
    df = preprocess_data(df, impute_features=USE_IMPUTATION)
    n_pre = len(df)
    df = feature_engineering(df)
    n_fe = len(df)
    X_train, X_val, X_test, y_train, y_val, y_test, _ = training_data_split(df)

    print("Data: load → preprocess → feature_eng")
    print(f"  Rows: {n_load} → {n_pre} (impute={USE_IMPUTATION}) → {n_fe}")
    print(f"  Features: {X_train.shape[1]} cols")
    print("Optuna: {} trials per tunable model.\n".format(N_TRIALS))

    results = []

    # No tuning
    dummy = DummyRegressor(strategy="mean")
    dummy.fit(X_train, y_train)
    results.append(
        (
            "Dummy (mean)",
            evaluate_model(dummy, X_val, y_val),
            evaluate_model(dummy, X_test, y_test),
        )
    )

    # Built-in CV 
    ridge = RidgeCV(alphas=np.logspace(-3, 2, 100), cv=5).fit(X_train, y_train)
    results.append(
        (
            "Ridge (CV)",
            evaluate_model(ridge, X_val, y_val),
            evaluate_model(ridge, X_test, y_test),
        )
    )

    ridge_log = RidgeCV(alphas=np.logspace(-3, 2, 100), cv=5)
    ridge_log.fit(X_train, np.log1p(y_train))

    def _ridge_log_predict(X):
        return np.expm1(ridge_log.predict(X))

    results.append(
        (
            "Ridge (log y)",
            evaluate_model(_PredictorWrapper(_ridge_log_predict), X_val, y_val),
            evaluate_model(_PredictorWrapper(_ridge_log_predict), X_test, y_test),
        )
    )

    enet = ElasticNetCV(
        cv=5,
        l1_ratio=[0.1, 0.5, 0.9, 1.0],
        alphas=np.logspace(-3, 1, 50),
        max_iter=5000,
    ).fit(X_train, y_train)
    results.append(
        (
            "ElasticNet (CV)",
            evaluate_model(enet, X_val, y_val),
            evaluate_model(enet, X_test, y_test),
        )
    )

    # Optuna-tuned models
    print("Tuning Random Forest...")
    _, vm, tm = tune_and_fit_rf(X_train, y_train, X_val, y_val, X_test, y_test)
    results.append(("Random Forest (tuned)", vm, tm))

    print("Tuning Gradient Boosting...")
    _, vm, tm = tune_and_fit_gbm(X_train, y_train, X_val, y_val, X_test, y_test)
    results.append(("Gradient Boosting (tuned)", vm, tm))

    print("Tuning Hist Gradient Boosting...")
    _, vm, tm = tune_and_fit_histgb(X_train, y_train, X_val, y_val, X_test, y_test)
    results.append(("Hist Gradient Boosting (tuned)", vm, tm))

    print("Tuning Extra Trees...")
    _, vm, tm = tune_and_fit_extra_trees(
        X_train, y_train, X_val, y_val, X_test, y_test
    )
    results.append(("Extra Trees (tuned)", vm, tm))

    print("Tuning XGBoost...")
    _, vm, tm = tune_and_fit_xgboost(
        X_train, y_train, X_val, y_val, X_test, y_test
    )
    results.append(("XGBoost (tuned)", vm, tm))

    print("Tuning SVR...")
    _, vm, tm = tune_and_fit_svr(X_train, y_train, X_val, y_val, X_test, y_test)
    results.append(("SVR (tuned)", vm, tm))

    print("Tuning FNN (MLP)...")
    _, vm, tm = tune_and_fit_mlp(X_train, y_train, X_val, y_val, X_test, y_test)
    results.append(("FNN (MLP) (tuned)", vm, tm))

    print("Tuning Keras...")
    _, vm, tm = tune_and_fit_keras(
        X_train, y_train, X_val, y_val, X_test, y_test
    )
    results.append(("Keras (tuned)", vm, tm))

    print("\nModel comparison (Validation | Test)")
    print("-" * 72)
    print(
        f"{'Model':<28} {'Val MAE':>8} {'Val RMSE':>8} {'Val R²':>8}  |  "
        f"{'Test MAE':>8} {'Test RMSE':>8} {'Test R²':>8}"
    )
    print("-" * 72)
    for name, vm, tm in results:
        print(
            f"{name:<28} "
            f"{vm['mae']:>8.4f} {vm['rmse']:>8.4f} {vm['r2']:>8.4f}  |  "
            f"{tm['mae']:>8.4f} {tm['rmse']:>8.4f} {tm['r2']:>8.4f}"
        )
    print("-" * 72)

    best_name, _, best_test = max(results, key=lambda r: r[2]["r2"])
    print(f"\nBest by Test R²: {best_name}  (R² = {best_test['r2']:.4f})")


if __name__ == "__main__":
    main()

Data: load → preprocess → feature_eng
  Rows: 700 → 700 (impute=True) → 693
  Features: 87 cols
Optuna: 25 trials per tunable model.



/opt/miniconda3/envs/research2/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.363e+00, tolerance: 2.349e+00
  model = cd_fast.enet_coordinate_descent_gram(
/opt/miniconda3/envs/research2/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.987e+01, tolerance: 2.349e+00
  model = cd_fast.enet_coordinate_descent_gram(
/opt/miniconda3/envs/research2/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or conside

Tuning Random Forest...
Tuning Gradient Boosting...
Tuning Hist Gradient Boosting...
Tuning Extra Trees...
Tuning XGBoost...
Tuning SVR...
Tuning FNN (MLP)...


/opt/miniconda3/envs/research2/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/miniconda3/envs/research2/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/miniconda3/envs/research2/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/miniconda3/envs/research2/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


Tuning Keras...

Model comparison (Validation | Test)
------------------------------------------------------------------------
Model                         Val MAE Val RMSE   Val R²  |  Test MAE Test RMSE  Test R²
------------------------------------------------------------------------
Dummy (mean)                   7.0965   8.3467  -0.4633  |    7.5699   9.6489  -0.5894
Ridge (CV)                     5.5423   6.8718   0.0081  |    6.9000   8.3325  -0.1853
Ridge (log y)                  5.9202   7.3301  -0.1286  |    7.6363   9.1598  -0.4324
ElasticNet (CV)                5.6748   7.0244  -0.0364  |    6.9117   8.3339  -0.1857
Random Forest (tuned)          5.6581   7.0770  -0.0520  |    6.2703   7.7640  -0.0291
Gradient Boosting (tuned)      5.7947   7.1437  -0.0719  |    6.5160   8.0957  -0.1189
Hist Gradient Boosting (tuned)   5.7331   7.0856  -0.0545  |    6.4274   7.9437  -0.0773
Extra Trees (tuned)            5.5260   6.9024  -0.0007  |    6.2844   7.8231  -0.0448
XGBoost (tuned